In [4]:
import os
import numpy as np
import torch
from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde

SEED = 42
NUM_SAMPLES = 2000
CACHE_DIR = "no-ml10m-tsne_cache/"
os.makedirs(CACHE_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)

# ======================================================
# 1. 读取四个 embedding（你已有）
# ======================================================
data_dict = {
    "u_sim": torch.from_numpy(np.load("no_cl_ml10m_raw_embeddings/ml10m_u_sim.npy")),
    "i_sim": torch.from_numpy(np.load("no_cl_ml10m_raw_embeddings/ml10m_i_sim.npy")),
    "u_agg": torch.from_numpy(np.load("no_cl_ml10m_raw_embeddings/ml10m_u_agg.npy")),
    "i_agg": torch.from_numpy(np.load("no_cl_ml10m_raw_embeddings/ml10m_i_agg.npy")),
}

In [5]:
def get_tsne_unit_projection(
    emb: torch.Tensor,
    name: str,
    num_samples: int = 2000,
    seed: int = 42,
    cache_dir: str = "tsne_cache",
):
    cache_path = os.path.join(cache_dir, f"{name}_tsne_unit.npz")

    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f"[Load cached t-SNE] {name}")
        return data["z_unit"]

    print(f"[Run sampling + t-SNE] {name}")
    rng = np.random.default_rng(seed)
    idx = rng.choice(emb.shape[0], size=num_samples, replace=False)
    sample = emb[idx].cpu().numpy()

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate=200,
        max_iter=3000,
        init="pca",
        random_state=seed
    )

    z_2d = tsne.fit_transform(sample)
    z_unit = z_2d / np.linalg.norm(z_2d, axis=1, keepdims=True)

    np.savez(cache_path, z_unit=z_unit)
    return z_unit

In [6]:
for idx, (name, emb) in enumerate(data_dict.items()):

    # ---------- t-SNE ----------
    z_unit = get_tsne_unit_projection(
        emb,
        name=name,
        num_samples=NUM_SAMPLES,
        seed=SEED,
        cache_dir=CACHE_DIR
    )

[Run sampling + t-SNE] u_sim
[Run sampling + t-SNE] i_sim
[Run sampling + t-SNE] u_agg
[Run sampling + t-SNE] i_agg
